In [ ]:
"""
Bees vs Ants Binary Classifier — Transfer Learning Foundation
================================================================
Production-grade binary image classifier built on ResNet18 with
transfer learning. The foundational project establishing the core
supervised classification pipeline applied at scale in later projects.

Architecture:
    - Backbone  : ResNet18, pretrained on ImageNet
    - Head      : Custom nn.Linear binary classifier
    - Optimizer : AdamW with weight decay
    - Precision : AMP (FP16 autocast + GradScaler)
    - Fine-tuning: Configurable — frozen backbone or full fine-tune

Dataset:
    PyTorch Hymenoptera dataset (~400 images of bees and ants).
    Small dataset size is the primary engineering constraint driving
    every architectural decision: small model, heavy regularization,
    aggressive data augmentation.

Deployment:
    FastAPI + Docker (CPU inference)

Author  : Alejandro Toro Arrabal
Version : 2.0.0
"""

# ── Standard Library
import copy
import logging
import os
import random
import sys
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Optional
from urllib.request import urlretrieve

# ── Third Party
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from torchvision import datasets, models, transforms
from tqdm import tqdm

In [ ]:
# ════════════════════════════════════════════════════════
# 1. LOGGING INFRASTRUCTURE
# ════════════════════════════════════════════════════════

def setup_logger(name: str = "BeesVSAnts_CV") -> logging.Logger:
    """Configures a structured stdout logger with duplicate handler prevention."""
    logger = logging.getLogger(name)
    if logger.hasHandlers():
        logger.handlers.clear()
    logger.setLevel(logging.INFO)
    logger.propagate = False  # Prevents Colab's hidden logger from duplicating output
    formatter = logging.Formatter(
        fmt="%(asctime)s - [%(levelname)s] - %(message)s",
        datefmt="%H:%M:%S"
    )
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    return logger


logger = setup_logger()

In [ ]:
# ════════════════════════════════════════════════════════
# 2. CONFIGURATION
# ════════════════════════════════════════════════════════

@dataclass(frozen=True)
class ExperimentConfig:
    """
    Immutable configuration for the Bees vs Ants classification pipeline.

    FINE_TUNE controls the transfer learning strategy:
        False -> backbone frozen, only classification head trains (fast,
                 low overfitting risk — appropriate for ~400 images)
        True  -> full network fine-tuned (higher capacity, higher
                 overfitting risk on this dataset size)
    """

    # ── Data
    DATA_URL: str = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
    DATA_DIR: Path = Path("hymenoptera_data")
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 2

    # ── Model
    NUM_CLASSES: int = 2
    FINE_TUNE: bool = False  # False = freeze feature extractor

    # ── Training
    EPOCHS: int = 25
    LEARNING_RATE: float = 1e-3
    WEIGHT_DECAY: float = 1e-4
    STEP_SIZE: int = 7
    GAMMA: float = 0.1
    SEED: int = 42

    # ── Checkpointing
    SAVE_PATH: str = "best_model.pth"

In [ ]:
# ════════════════════════════════════════════════════════
# 3. BEE/ANT CLASSIFIER — OOP CONTROLLER
# ════════════════════════════════════════════════════════

class BeeAntClassifier:
    """
    End-to-end binary classification pipeline for the Hymenoptera dataset.

    Responsibilities:
        - Dataset download and DataLoader construction with full
          hardware optimization (pinned memory, prefetching, persistent workers)
        - ResNet18 transfer learning (frozen or fine-tuned backbone)
        - AMP-accelerated training (autocast + GradScaler)
        - Checkpointing on best validation accuracy
        - API-ready inference (accepts numpy arrays, returns confidence scores)
        - Per-class evaluation (confusion matrix, precision/recall/F1)
        - Visual inspection grid and production smoke testing

    Usage:
        config = ExperimentConfig(EPOCHS=25)
        classifier = BeeAntClassifier(config)
        classifier.prepare_data()
        classifier.build_model()
        classifier.train()
    """

    def __init__(self, config: ExperimentConfig) -> None:
        self.cfg = config
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        self._set_seed()
        logger.info(f"System initialized on: {self.device}")

        # ── Training transforms — stochastic augmentation for generalization
        # Separate pipelines for train/val prevent data leakage and improve
        # generalization via controlled, train-only augmentation.
        self.train_transform = transforms.Compose([
            transforms.RandomResizedCrop(224),      # Aspect ratio + scale invariance
            transforms.RandomHorizontalFlip(),       # Symmetry invariance
            transforms.RandomRotation(15),            # Rotation invariance (+/-15 degrees)
            transforms.ColorJitter(brightness=0.1, contrast=0.1),  # Lighting robustness
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        # ── Validation transforms — deterministic, MUST match inference exactly
        # CRITICAL: predict_image() reuses this exact transform object below —
        # never redefine it inline elsewhere, or inference and validation
        # preprocessing can silently drift apart.
        self.val_transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        # State placeholders
        self.model: Optional[nn.Module] = None
        self.dataloaders: Dict = {}
        self.dataset_sizes: Dict = {}
        self.class_names: list = []
        self.scaler = None

    # ────────────────────────────────────────────────────
    # PRIVATE UTILITIES
    # ────────────────────────────────────────────────────

    def _set_seed(self) -> None:
        """
        Locks all random number generators for full reproducibility.

        Required components: PYTHONHASHSEED, random, numpy, torch,
        torch.cuda (all GPUs), cudnn.deterministic, cudnn.benchmark=False.
        """
        seed = self.cfg.SEED
        os.environ["PYTHONHASHSEED"] = str(seed)
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

    # ────────────────────────────────────────────────────
    # PUBLIC PIPELINE METHODS
    # ────────────────────────────────────────────────────

    def prepare_data(self) -> None:
        """Downloads (if needed) and constructs optimized DataLoaders."""
        if not self.cfg.DATA_DIR.exists():
            try:
                logger.info("Downloading dataset...")
                zip_path = "data_temp.zip"
                urlretrieve(self.cfg.DATA_URL, zip_path)
                with zipfile.ZipFile(zip_path, "r") as zip_ref:
                    zip_ref.extractall(".")
                os.remove(zip_path)
                logger.info("Dataset extracted.")
            except Exception as e:
                logger.error(f"Dataset download failed: {e}")
                raise RuntimeError(f"Failed to download/extract dataset: {e}") from e
        else:
            logger.info("Dataset found. Skipping download.")

        image_datasets = {
            "train": datasets.ImageFolder(str(self.cfg.DATA_DIR / "train"), self.train_transform),
            "val": datasets.ImageFolder(str(self.cfg.DATA_DIR / "val"), self.val_transform),
        }

        self.dataloaders = {
            x: torch.utils.data.DataLoader(
                image_datasets[x],
                batch_size=self.cfg.BATCH_SIZE,
                shuffle=(x == "train"),
                num_workers=self.cfg.NUM_WORKERS,
                pin_memory=True,            # Locks tensors for fast PCIe transfer
                prefetch_factor=2,           # CPU pre-loads 2 batches ahead of GPU
                persistent_workers=True,     # Avoids worker respawn overhead per epoch
            )
            for x in ["train", "val"]
        }

        self.dataset_sizes = {x: len(image_datasets[x]) for x in ["train", "val"]}
        self.class_names = image_datasets["train"].classes

        logger.info(f"Data loaded. Classes: {self.class_names}")

    def build_model(self) -> None:
        """
        Loads ResNet18 and adapts the head for binary classification.

        AdamW: decouples weight decay from gradient scaling — unlike
        standard Adam, which corrupts the regularization effect.
        """
        logger.info("Building model architecture...")

        is_gpu = torch.cuda.is_available()
        self.scaler = torch.amp.GradScaler("cuda" if is_gpu else "cpu", enabled=is_gpu)

        weights = models.ResNet18_Weights.IMAGENET1K_V1
        self.model = models.resnet18(weights=weights)

        if not self.cfg.FINE_TUNE:
            for param in self.model.parameters():
                param.requires_grad = False

        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, self.cfg.NUM_CLASSES)
        self.model = self.model.to(self.device)

        self.criterion = nn.CrossEntropyLoss()

        params_to_update = [p for p in self.model.parameters() if p.requires_grad]
        self.optimizer = torch.optim.AdamW(
            params_to_update,
            lr=self.cfg.LEARNING_RATE,
            weight_decay=self.cfg.WEIGHT_DECAY,
        )
        self.scheduler = torch.optim.lr_scheduler.StepLR(
            self.optimizer, step_size=self.cfg.STEP_SIZE, gamma=self.cfg.GAMMA
        )

        mode = "fine-tuning full network" if self.cfg.FINE_TUNE else "frozen backbone (head only)"
        logger.info(f"Model ready. Training mode: {mode}.")

    def train(self) -> nn.Module:
        """Executes the AMP-accelerated training loop with checkpointing."""
        if self.model is None:
            raise RuntimeError("Model not built. Call build_model() first.")

        since = time.time()
        best_model_wts = copy.deepcopy(self.model.state_dict())
        best_acc = 0.0

        logger.info(f"Starting training for {self.cfg.EPOCHS} epochs...")

        for epoch in range(self.cfg.EPOCHS):
            logger.info(f"--- Epoch {epoch + 1}/{self.cfg.EPOCHS} ---")

            for phase in ["train", "val"]:
                self.model.train() if phase == "train" else self.model.eval()

                running_loss = 0.0
                running_corrects = 0

                pbar = tqdm(self.dataloaders[phase], leave=False, desc=f"{phase}")
                for inputs, labels in pbar:
                    inputs = inputs.to(self.device)
                    labels = labels.to(self.device)
                    self.optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == "train"):
                        # AMP forward pass — eligible ops cast to FP16
                        with torch.amp.autocast(device_type=self.device.type):
                            outputs = self.model(inputs)
                            _, preds = torch.max(outputs, 1)
                            loss = self.criterion(outputs, labels)

                        if phase == "train":
                            # GradScaler prevents FP16 gradient underflow
                            self.scaler.scale(loss).backward()
                            self.scaler.step(self.optimizer)
                            self.scaler.update()

                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels.data)
                    pbar.set_postfix(loss=loss.item())

                if phase == "train":
                    self.scheduler.step()

                epoch_loss = running_loss / self.dataset_sizes[phase]
                epoch_acc = running_corrects.double() / self.dataset_sizes[phase]

                logger.info(f"{phase.upper()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

                if phase == "val" and epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(self.model.state_dict())
                    logger.info(f"New best model! Acc: {best_acc:.4f}")

        time_elapsed = time.time() - since
        logger.info(f"Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
        logger.info(f"Best Val Acc: {best_acc:.4f}")

        self.model.load_state_dict(best_model_wts)
        return self.model

    def load_model(self, path: str) -> None:
        """Loads weights from disk and runs warmup inference."""
        if not os.path.exists(path):
            raise FileNotFoundError(f"Model file not found: {path}")

        if self.model is None:
            self.build_model()

        self.model.load_state_dict(torch.load(path, map_location=self.device))
        logger.info(f"Model loaded successfully from: {path}")

        # Warmup pass — compiles CUDA kernels before first real request
        logger.info("Running warmup inference...")
        dummy = torch.zeros(1, 3, 224, 224).to(self.device)
        with torch.inference_mode():
            self.model(dummy)
        logger.info("Model warmed up and ready.")

    def save_model(self, path: str) -> None:
        """Persists model state dict to disk — crucial for the train-once, deploy-many MLOps pattern."""
        if self.model is None:
            logger.warning("No model to save!")
            return
        torch.save(self.model.state_dict(), path)
        logger.info(f"Model weights saved to: {path}")

    def predict_image(self, image: np.ndarray) -> Dict[str, Any]:
        """
        API-ready inference method. Accepts numpy arrays from FastAPI's
        in-memory byte decoding pipeline (cv2.imdecode output).

        CRITICAL: reuses self.val_transform directly rather than redefining
        preprocessing inline. Any divergence between training validation
        preprocessing and inference preprocessing causes silently degraded
        production accuracy.

        Args:
            image: RGB numpy array of shape (H, W, 3), dtype uint8.
                   Note: OpenCV reads BGR — convert before passing:
                   image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        Returns:
            JSON-serializable dict:
            {
                "prediction": str,      # "ants" or "bees"
                "confidence": float,    # softmax probability [0, 1]
                "class_id": int
            }

        Raises:
            ValueError    : If image is None or empty.
            RuntimeError  : If model is not loaded.
        """
        if image is None or image.size == 0:
            raise ValueError(
                "Invalid image — array is None or empty. "
                "Verify the byte decoding step produced a valid array."
            )
        if self.model is None:
            raise RuntimeError("Model not loaded. Call load_model() before predict_image().")

        pil_image = Image.fromarray(image).convert("RGB")
        img_tensor = self.val_transform(pil_image).unsqueeze(0).to(self.device)

        self.model.eval()
        with torch.inference_mode():
            outputs = self.model(img_tensor)
            probs = torch.softmax(outputs, dim=1)
            confidence, pred_idx = probs.max(dim=1)

        return {
            "prediction": self.class_names[int(pred_idx[0])],
            "confidence": round(float(confidence[0]), 4),
            "class_id"  : int(pred_idx[0]),
        }

    def evaluate(self) -> Dict[str, Any]:
        """
        Per-class evaluation beyond aggregate accuracy.

        For a small ~400-image dataset, aggregate accuracy can hide
        meaningful asymmetry — e.g. the model may classify ants near-perfectly
        but struggle specifically with bees. Confusion matrix and per-class
        precision/recall/F1 surface that asymmetry directly.
        """
        if self.model is None:
            raise RuntimeError("Model not loaded.")

        self.model.eval()
        all_preds, all_labels = [], []

        with torch.inference_mode():
            for inputs, labels in self.dataloaders["val"]:
                inputs = inputs.to(self.device)
                outputs = self.model(inputs)
                preds = outputs.argmax(dim=1).cpu()
                all_preds.extend(preds.numpy())
                all_labels.extend(labels.numpy())

        cm = confusion_matrix(all_labels, all_preds)
        report = classification_report(
            all_labels, all_preds, target_names=self.class_names, output_dict=True
        )

        logger.info(f"Confusion Matrix:\n{cm}")
        logger.info(
            classification_report(all_labels, all_preds, target_names=self.class_names)
        )

        return {"confusion_matrix": cm.tolist(), "classification_report": report}

    def smoke_test(self) -> bool:
        """
        Validates the inference pipeline before deployment.

        Tests: load process integrity, dummy forward pass, and output
        shape/probability sanity — not prediction correctness, since a
        random tensor has no ground truth to compare against.
        """
        if self.model is None:
            logger.error("Smoke test failed: model not loaded.")
            return False

        try:
            dummy_input = torch.randn(1, 3, 224, 224).to(self.device)
            self.model.eval()

            with torch.inference_mode():
                output = self.model(dummy_input)
                probs = torch.softmax(output, dim=1)

            assert output.shape == (1, self.cfg.NUM_CLASSES), \
                f"Expected output shape (1, {self.cfg.NUM_CLASSES}), got {output.shape}"
            assert torch.allclose(probs.sum(), torch.tensor(1.0), atol=1e-5), \
                "Probabilities do not sum to 1"

            logger.info(f"Smoke test passed. Output shape: {output.shape}")
            return True

        except Exception as e:
            logger.error(f"Smoke test failed: {e}")
            return False

    def visualize(self, num_images: int = 6) -> None:
        """Inference and visualization grid — qualitative inspection tool."""
        if self.model is None:
            logger.error("Model not loaded. Cannot visualize.")
            return

        was_training = self.model.training
        self.model.eval()
        images_so_far = 0

        plt.figure(figsize=(10, 10))

        with torch.inference_mode():
            for inputs, labels in self.dataloaders["val"]:
                inputs = inputs.to(self.device)
                labels = labels.to(self.device)

                outputs = self.model(inputs)
                _, preds = torch.max(outputs, 1)

                for j in range(inputs.size()[0]):
                    images_so_far += 1
                    ax = plt.subplot(num_images // 2, 2, images_so_far)
                    ax.axis("off")

                    img = inputs[j].cpu().numpy().transpose((1, 2, 0))
                    mean = np.array([0.485, 0.456, 0.406])
                    std = np.array([0.229, 0.224, 0.225])
                    img = std * img + mean
                    img = np.clip(img, 0, 1)

                    ax.set_title(f"Predicted: {self.class_names[preds[j]]}")
                    ax.imshow(img)

                    if images_so_far == num_images:
                        self.model.train(mode=was_training)
                        plt.show()
                        return

        self.model.train(mode=was_training)

In [ ]:
# ════════════════════════════════════════════════════════
# 4. EXECUTION ENTRYPOINT
# ════════════════════════════════════════════════════════

if __name__ == "__main__":

    config = ExperimentConfig(EPOCHS=25)
    classifier = BeeAntClassifier(config)

    # ── Phase 1: Training (the "Factory" phase)
    logger.info("Starting Factory...")
    classifier.prepare_data()
    classifier.build_model()
    classifier.train()
    classifier.save_model(config.SAVE_PATH)

    # ── Phase 2: Per-class evaluation
    classifier.evaluate()

    # ── Phase 3: Production validation (the "Quality Control" phase)
    # Simulates what Docker will do: start fresh, load the saved file
    logger.info("Starting Quality Control...")
    production_model = BeeAntClassifier(config)
    production_model.load_model(config.SAVE_PATH)

    if production_model.smoke_test():
        logger.info("MODEL IS READY FOR DOCKER.")
    else:
        logger.error("MODEL DEFECTIVE. DO NOT DEPLOY.")